In [80]:
from pathlib import Path
from metasmith.python_api import Agent, Source, Std, DataInstanceLibrary, TransformInstanceLibrary, WorkflowTask
from metasmith.python_api import DataTypeLibrary, Endpoint
from local.constants import WORKSPACE_ROOT

# dtypes, containers, transforms = Std()

path_to_agent_home = Path("./cache/local_home").resolve()
smith = Agent(
    home = Source.FromLocal(path_to_agent_home),
)
# smith.Deploy()

In [81]:
mock_types = DataTypeLibrary(types=dict(
    a=Endpoint({"test", "a"}),
    b=Endpoint({"test", "b"}),
    c=Endpoint({"test", "c"}),
))

transforms = TransformInstanceLibrary("./transforms/simple_1", include_std=False)
transforms.AddTypeLibrary("mock", mock_types)
transforms.AddStub("no_op")
transforms.Save()

In [82]:
inputs = DataInstanceLibrary("./cache/dev21.mock.xgdb")
samples = []
for i in range(10):
    in_path = WORKSPACE_ROOT/f"main/local_mock/cache/test/mock_d.{i}"
    in_path.parent.mkdir(exist_ok=True)
    with open(in_path, "w") as f:
        f.write("10")
    inputs.AddTypeLibrary("mock", mock_types)
    inputs.AddItem(in_path, "mock::a")
    samples.append(in_path)
inputs.Save()
for p, n, e in inputs.Iterate():
    print(n, e, e.parents)

mock::a <[a,test]:q7XtOGUL> set()
mock::a <[a,test]:q7XtOGUL> set()
mock::a <[a,test]:q7XtOGUL> set()
mock::a <[a,test]:q7XtOGUL> set()
mock::a <[a,test]:q7XtOGUL> set()
mock::a <[a,test]:q7XtOGUL> set()
mock::a <[a,test]:q7XtOGUL> set()
mock::a <[a,test]:q7XtOGUL> set()
mock::a <[a,test]:q7XtOGUL> set()
mock::a <[a,test]:q7XtOGUL> set()


In [83]:
for loc, t,  in transforms.IterateTransforms():
    print(t.model)

{a-test}->{b-test}


In [84]:
task = smith.GenerateWorkflow(
    samples    = [inputs.AsView({p}) for p in samples],
    resources  = [],
    transforms = [transforms],
    targets    = [mock_types["b"]]
)
with open(WORKSPACE_ROOT/"secrets/slurm_account_fir") as f:
    SLURM_ACCOUNT = f.read()

task.config = dict(
    nextflow = dict(
        preset="slurm",
        slurm_account=SLURM_ACCOUNT,
        cpus=1,
        array=10,
        queueSize=500,
        memory=8,
        time=3,
    )
)
print(task.GetKey())

v0kSNUO9


In [ ]:
smith.StageWorkflow(task, on_exist='clear', verify_external_paths=False)

2025-11-18_13-27-34  | connecting to deployed agent
2025-11-18_13-27-34  | starting relay service
 | > 2025-11-18_13-27-36  | starting relay server at [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/relay/XPS-laptop]
 | > 2025-11-18_13-27-36  | pid [65029]
 | > 2025-11-18_13-27-36  | success
2025-11-18_13-27-36 W| task already staged at [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/v0kSNUO9]
2025-11-18_13-27-36 W| clearing previously staged task
2025-11-18_13-27-36  | sending metadata for workflow [v0kSNUO9]
2025-11-18_13-27-38  | staging
2025-11-18_13-27-38  | external binds ['/home/tony/workspace/tools/Metasmith/main/local_mock/cache']
 | > including dev binds
 | > binds [--bind /home/tony/workspace/tools/Metasmith/main/local_mock/cache:/home/tony/workspace/tools/Metasmith/main/local_mock/cache --bind ./:/ws,/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home:/msm_home,/home/tony/.globus:/home/tony/.globus,/home/t

In [86]:
smith.RunWorkflow(task)

2025-11-18_13-27-41  | connecting to deployed agent
2025-11-18_13-27-41  | starting relay service
 | > 2025-11-18_13-27-43 W| relay server already running at [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/relay/XPS-laptop]
2025-11-18_13-27-43  | triggering execution of [v0kSNUO9]
2025-11-18_13-27-43  | external binds ['/home/tony/workspace/tools/Metasmith/main/local_mock/cache']
2025-11-18_13-27-43  | closing connection


In [8]:
assert False

AssertionError: 

In [79]:
import pandas as pd
import json
from datetime import datetime

with open("./cache/nxf_report.dev21.html") as f:
    found = False
    for l in f:
        if l.strip().startswith('window.data = { "trace":['): 
            found = True
            continue
        if not found: 
            continue

        d = json.loads('{ "trace":[' + l[:-2])
        break
dfo = pd.DataFrame(d['trace'])
df = dfo
df = df[df.attempt == "3"]
df = df[~df.status.isin({"COMPLETED", "ABORTED"})]

df = df['hash, status, exit, submit, peak_rss, peak_vmem, duration, cpu_model, error_action, attempt'.split(', ')]
start = datetime.fromtimestamp(0)
df['duration'] = df.duration.apply(lambda x: (datetime.fromtimestamp(int(x)/1000.0)-start))
df

,hash,status,exit,submit,peak_rss,peak_vmem,duration,cpu_model,error_action,attempt
154,11/e42f00,FAILED,140,1763468325949,-,-,0 days 02:59:29.396000,-,RETRY,3


In [53]:
df.columns

Index(['task_id', 'hash', 'native_id', 'process', 'module', 'container', 'tag',
       'name', 'status', 'exit', 'submit', 'start', 'complete', 'duration',
       'realtime', '%cpu', '%mem', 'rss', 'vmem', 'peak_rss', 'peak_vmem',
       'rchar', 'wchar', 'syscr', 'syscw', 'read_bytes', 'write_bytes',
       'attempt', 'workdir', 'script', 'scratch', 'queue', 'cpus', 'memory',
       'disk', 'time', 'env', 'error_action', 'vol_ctxt', 'inv_ctxt',
       'hostname', 'cpu_model'],
      dtype='object')

In [64]:
dfo.time.

0       10800000
1       10800000
2       10800000
3       10800000
4       10800000
         ...    
158    172800000
159    172800000
160    172800000
161    172800000
162    388800000
Name: time, Length: 163, dtype: object